# Build the complete cell
Assembles a **map-level human cell** you can explore, perturb, and infect — using the three models in the roles each is actually good at.

| model | role in the cell | why it, not another |
|---|---|---|
| **our integrated model** | identity, importance, compartment, wiring (the map itself) | only model that works for essentiality (AUC 0.86 vs Geneformer's random 0.53) |
| **atlas / scGPT cell-type layer** | which sub-network is live in *this* cell type (context) | same genome → many cells; supplies master TFs + active sets |
| **Geneformer** | direction & reach of a knockout's effect (dynamics-lite) | its genuine strength is *in-silico perturbation*, not static importance |

See `docs/CELL_ARCHITECTURE.md` for the full honest breakdown. This notebook runs end-to-end on **CPU**; the Geneformer step is **optional** and only *enriches* the cascade — the cell is complete without it.


In [ ]:
# [1] get the repo (Colab) — skip if running locally in the repo
import os, sys
if not os.path.exists('colab/build_cell_complete.py'):
    !git clone -q --branch claude/vectorize-gex-propensity-NRqBW https://github.com/nikku03/cell.git
    os.chdir('cell')
print('cwd:', os.getcwd())


In [ ]:
# [2] (optional) mount Drive to persist outputs
try:
    from google.colab import drive; drive.mount('/content/drive')
    PROJ='/content/drive/MyDrive/cell_model'; os.makedirs(PROJ, exist_ok=True)
except Exception:
    PROJ=None
print('persist ->', PROJ or 'local repo only')


## Model 1 — our integrated backbone (identity & importance)
The 14-layer table is committed in the repo. It is the spine every later step decorates.


In [ ]:
import pandas as pd, numpy as np, json
bb=pd.read_csv('outputs/orphan/integrated_cell_human.csv')
print('backbone genes:', len(bb), '| layers:', list(bb.columns))
print('essential labeled:', int(bb.essential.notna().sum()), '| TFs:', int((bb.is_tf==1).sum()))


## Model 2 — atlas / scGPT cell-type layer (context)
Turns the *static* backbone into a *cell*: which master TFs are on, hence which sub-network is live.
If you produced `celltype_expression.csv` from the ensemble notebook (Tabula Sapiens per-cell-type means),
drop it in `outputs/orphan/` or Drive and it will be used; otherwise the validated master-TF sets in
`build_cell_complete.py` are used (hepatocyte→HNF4A, cardiac→GATA4/TBX5, NK→EOMES, …).


In [ ]:
cte=None
for p in ['outputs/orphan/celltype_expression.csv', (PROJ+'/celltype_expression.csv') if PROJ else '']:
    if p and os.path.exists(p):
        cte=pd.read_csv(p, index_col=0); print('using measured cell-type expression:', cte.shape); break
if cte is None:
    print('no celltype_expression.csv found -> using validated master-TF sets (already in the builder). This is fine.')


## Model 3 — Geneformer in-silico perturbation (optional, dynamics-lite)
Geneformer's real strength: delete a gene from a cell's rank-encoding and read how the rest shifts —
a *direction of effect*. We use it only to **enrich** the knockout cascade, never for importance
(it scored ~random there). This cell is complete without it; run only if you want richer cascades.


In [ ]:
USE_GENEFORMER=False  # set True on a GPU runtime to enrich cascades
gf_perturb={}
if USE_GENEFORMER:
    try:
        # frozen forward pass only — no fine-tuning. See ensemble notebook for the loader.
        from transformers import AutoModel  # noqa
        print('Geneformer available — (enrichment hook; wire to your in-silico perturbation run)')
        # gf_perturb[gene] = [downstream genes with largest rank shift under KO]
    except Exception as e:
        print('Geneformer not set up, skipping enrichment:', e)
else:
    print('Geneformer enrichment OFF — cascade uses measured reg+PPI graph (deterministic, auditable).')


## Assemble → build → serve
`build_cell_complete.py` fuses the three roles + measured data (compartments, CollecTRI/DoRothEA reg,
STRING PPI, Reactome, HIV interactions, curated reactions, dark genes) into `cell_complete.json`.
`build_cell_app_complete.py` renders the self-contained interactive HTML.


In [ ]:
!python colab/build_cell_complete.py
!python colab/build_cell_app_complete.py
sz=os.path.getsize('outputs/orphan/cell_complete.html')//1024
print('built cell_complete.html:', sz, 'KB')
if PROJ:
    import shutil; shutil.copy('outputs/orphan/cell_complete.html', PROJ+'/cell_complete.html'); print('copied to Drive')


In [ ]:
# [serve] localhost link. In Colab this renders inline; locally it opens a browser.
IN_COLAB='google.colab' in sys.modules
if IN_COLAB:
    from IPython.display import HTML, display
    html=open('outputs/orphan/cell_complete.html').read()
    display(HTML(f'<iframe srcdoc="{html.replace(chr(34),chr(38)+chr(35)+chr(51)+chr(52)+chr(59))}" width=100% height=760 style=border:0></iframe>'))
    # or download: from google.colab import files; files.download('outputs/orphan/cell_complete.html')
else:
    print('run:  python colab/serve_cell.py   ->  http://localhost:8000/cell')


## What you can now do in the cell
- **Explore** — click any protein → its full trafficking journey (gene→hnRNA→mRNA→export→ribosome/ER→Golgi→destination) + regulators/targets/binders.
- **Processes / Metabolism** — color by function; trace core reactions substrate→product.
- **Remove/Mutate** — knock out a protein → cascade over the network → survives or inviable.
- **Dark genes** — the 5006-protein function frontier, placed by compartment + network.
- **cell type** — same genome, different master-TF active network.
- **Infect: HIV** — hijacked host machinery + HIV's host-dependency weak points (drug targets).
